In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [1]:
class GaleShapley:
    def __init__(self, residents_pref, hospitals_pref):
        """
        Initializing with preference dictionaries.
        residents_pref: dict {resident: [list of hospitals]}
        hospitals_pref: dict {hospital: [list of residents]}
        """
        self.r_pref = residents_pref
        self.h_pref = hospitals_pref
        self.matches = {}  # Stores {hospital: resident}
        self.free_residents = list(residents_pref.keys())

    def solve(self):
        # Continuing while there is a free resident who still has options
        while self.free_residents:
            r = self.free_residents.pop(0)
            r_list = self.r_pref[r]

            # If resident has run out of hospitals to propose to
            if not r_list:
                continue

            # Proposing to the first hospital on the list
            h = r_list.pop(0) 

            if h not in self.matches:
                # Hospital is free: Accepting immediately
                self.matches[h] = r
                print(f"{r} proposed to {h} -> Accepted (Free)")
            else:
                # Hospital is occupied: Comparing current vs new
                current_r = self.matches[h]
                h_list = self.h_pref[h]
                
                # Checking indices (lower index = higher preference)
                if h_list.index(r) < h_list.index(current_r):
                    # New resident is better
                    self.matches[h] = r
                    self.free_residents.append(current_r) # Old resident becomes free
                    print(f"{r} proposed to {h} -> Accepted (Swap: {current_r} rejected)")
                else:
                    # Current resident is better
                    self.free_residents.append(r) # Proposer remains free
                    print(f"{r} proposed to {h} -> Rejected")

        return self.matches
#Example use case
residents = {
    'R1': ['H1', 'H2', 'H3'],
    'R2': ['H2', 'H1', 'H3'],
    'R3': ['H1', 'H2', 'H3']
}
hospitals = {
    'H1': ['R2', 'R3', 'R1'],
    'H2': ['R1', 'R2', 'R3'],
    'H3': ['R1', 'R2', 'R3']
}

system = GaleShapley(residents, hospitals)
final_allocation = system.solve()
print("\nFinal Stable Matching:", final_allocation)

R1 proposed to H1 -> Accepted (Free)
R2 proposed to H2 -> Accepted (Free)
R3 proposed to H1 -> Accepted (Swap: R1 rejected)
R1 proposed to H2 -> Accepted (Swap: R2 rejected)
R2 proposed to H1 -> Accepted (Swap: R3 rejected)
R3 proposed to H2 -> Rejected
R3 proposed to H3 -> Accepted (Free)

Final Stable Matching: {'H1': 'R2', 'H2': 'R1', 'H3': 'R3'}
